In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
import pandas as pd

from config import DATA_INTERMEDIATE

# String paths (with trailing slash) keep the `path + "file.csv"` cells below working.
input_path = f"{DATA_INTERMEDIATE}/"
output_path = f"{DATA_INTERMEDIATE}/"

In [ ]:
deitytext = pd.read_csv(input_path + "corpus_cleaned_subjects.csv", encoding="utf-8-sig")

# Create a column to preserve original order for tie-breaking
deitytext["original_order"] = range(len(deitytext))

# Sort by Primary_Author, Title, Published, Page (all ascending),
# then by original_order to preserve order when Page values are the same
deitytext_sorted = deitytext.sort_values(
    by=["Primary_Author", "Title", "Published", "Page", "original_order"],
    ascending=[True, True, True, True, True],
)

# Drop the helper column
deitytext_sorted = deitytext_sorted.drop("original_order", axis=1)

# Reset index if desired
deitytext_sorted = deitytext_sorted.reset_index(drop=True)

# Save to CSV
deitytext_sorted.to_csv(
    output_path + "processed_output_11s_sorted.csv", index=False, encoding="utf-8-sig"
)

print(f"Sorted dataframe shape: {deitytext_sorted.shape}")
print("\nFirst few rows:")
print(deitytext_sorted.head())

In [ ]:
# Get unique combinations of Primary_Author, Title, Published
unique_combinations = deitytext_sorted[["Primary_Author", "Title", "Published"]].drop_duplicates()

print(f"Total unique author/title/published combinations: {len(unique_combinations)}")

# Randomly sample from these unique combinations
# Adjust the sample size as needed - here I'm using 100 as an example
sample_size = 100  # Change this to your desired number

# If you want a specific number:
sampled_combinations = unique_combinations.sample(
    n=min(sample_size, len(unique_combinations)), random_state=42
)

# Or if you want a fraction (e.g., 10% of unique combinations):
# sampled_combinations = unique_combinations.sample(frac=0.1, random_state=42)

# Merge back to get all rows associated with the sampled combinations
deitytext_sample = deitytext_sorted.merge(
    sampled_combinations, on=["Primary_Author", "Title", "Published"], how="inner"
)

print(f"Sample dataframe shape: {deitytext_sample.shape}")
print(
    f"Unique combinations in sample: {deitytext_sample[['Primary_Author', 'Title', 'Published']].drop_duplicates().shape[0]}"
)

# Save the sample
deitytext_sample.to_csv(output_path + "processed_sample.csv", index=False, encoding="utf-8-sig")

print("\nFirst few rows of sample:")
print(deitytext_sample.head())

In [ ]:
import matplotlib.pyplot as plt

# Count how many rows are in each unique combination in the COMPLETE dataset
combination_counts_full = deitytext_sorted.groupby(["Primary_Author", "Title", "Published"]).size()

print("Statistics on rows per combination (COMPLETE dataset):")
print(combination_counts_full.describe())

# Create a histogram
plt.figure(figsize=(12, 6))
plt.hist(combination_counts_full, bins=1000, edgecolor="black", alpha=0.7)
plt.xlabel("Number of Rows per Unique Combination", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.xscale("log")
plt.title(
    "Distribution of Row Counts per Author/Title/Published Combination (Complete Dataset)",
    fontsize=14,
)
plt.grid(axis="y", alpha=0.3)

# Add some statistics to the plot
mean_val = combination_counts_full.mean()
median_val = combination_counts_full.median()
plt.axvline(mean_val, color="red", linestyle="--", linewidth=2, label=f"Mean: {mean_val:.1f}")
plt.axvline(
    median_val, color="green", linestyle="--", linewidth=2, label=f"Median: {median_val:.1f}"
)
plt.legend()

plt.tight_layout()
plt.savefig(output_path + "combination_distribution_full.png", dpi=300, bbox_inches="tight")
plt.show()

# Optional: Create a bar chart for the top 20 combinations
plt.figure(figsize=(14, 8))
top_20_full = combination_counts_full.nlargest(20)
# Create labels for the x-axis
labels = [
    f"{auth[:20]}...\n{title[:30]}..." if len(auth) > 20 or len(title) > 30 else f"{auth}\n{title}"
    for auth, title, _ in top_20_full.index
]


print("\nTop 10 combinations with most rows (COMPLETE dataset):")
print(top_20_full.head(10))

# Optional: Compare sample vs full dataset
print("\n=== COMPARISON ===")
print(f"Total unique combinations in full dataset: {len(combination_counts_full)}")
print(f"Total rows in full dataset: {len(deitytext_sorted)}")
print(f"Total rows in sample: {len(deitytext_sample)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Count how many rows are in each unique combination in the COMPLETE dataset
combination_counts_full = deitytext_sorted.groupby(["Primary_Author", "Title", "Published"]).size()

# Sort the counts for CDF
sorted_counts = np.sort(combination_counts_full.values)
# Calculate cumulative probabilities
cumulative_prob = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts)

# Create CDF plot with log scale
plt.figure(figsize=(12, 6))
plt.plot(sorted_counts, cumulative_prob, linewidth=2)
plt.xscale("log")  # Set x-axis to log scale
plt.xlabel("Number of Rows per Unique Combination (log scale)", fontsize=12)
plt.ylabel("Cumulative Probability", fontsize=12)
plt.title("CDF of Row Counts per Author/Title/Published Combination (Log Scale)", fontsize=14)
plt.grid(True, alpha=0.3, which="both")  # Show grid for both major and minor ticks

# Add percentile lines
mean_val = combination_counts_full.mean()
median_val = combination_counts_full.median()
percentile_75 = combination_counts_full.quantile(0.75)
percentile_90 = combination_counts_full.quantile(0.90)
percentile_95 = combination_counts_full.quantile(0.95)
percentile_99 = combination_counts_full.quantile(0.99)

plt.axvline(
    median_val,
    color="green",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"Median (50%): {median_val:.1f}",
)
plt.axvline(
    percentile_75,
    color="orange",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"75th percentile: {percentile_75:.1f}",
)
plt.axvline(
    percentile_90,
    color="red",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"90th percentile: {percentile_90:.1f}",
)
plt.axvline(
    percentile_95,
    color="darkred",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"95th percentile: {percentile_95:.1f}",
)
plt.axvline(
    percentile_99,
    color="purple",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"99th percentile: {percentile_99:.1f}",
)
# Add horizontal lines at key probabilities
plt.axhline(0.5, color="green", linestyle=":", linewidth=1, alpha=0.5)
plt.axhline(0.75, color="orange", linestyle=":", linewidth=1, alpha=0.5)
plt.axhline(0.90, color="red", linestyle=":", linewidth=1, alpha=0.5)
plt.axhline(0.95, color="darkred", linestyle=":", linewidth=1, alpha=0.5)
plt.axhline(0.99, color="purple", linestyle=":", linewidth=1, alpha=0.5)

plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(output_path + "combination_counts_cdf_log.png", dpi=300, bbox_inches="tight")
plt.show()

# Print some useful statistics from the CDF
print("\nCDF Statistics:")
print(f"Median (50th percentile): {median_val:.1f} rows")
print(f"75th percentile: {percentile_75:.1f} rows")
print(f"90th percentile: {percentile_90:.1f} rows")
print(f"95th percentile: {percentile_95:.1f} rows")
print(f"99th percentile: {percentile_99:.1f} rows")
print(f"\n{(cumulative_prob[sorted_counts <= 10].max() * 100):.1f}% of combinations have ≤10 rows")
print(f"{(cumulative_prob[sorted_counts <= 20].max() * 100):.1f}% of combinations have ≤20 rows")
print(f"{(cumulative_prob[sorted_counts <= 50].max() * 100):.1f}% of combinations have ≤50 rows")
print(f"{(cumulative_prob[sorted_counts <= 100].max() * 100):.1f}% of combinations have ≤100 rows")

In [ ]:
import matplotlib.pyplot as plt

# Count how many rows are in each unique combination in the sample
combination_counts = deitytext_sample.groupby(["Primary_Author", "Title", "Published"]).size()

print("Statistics on rows per combination:")
print(combination_counts.describe())

# Create a histogram
plt.figure(figsize=(12, 6))
plt.hist(combination_counts, bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("Number of Rows per Unique Combination", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.title("Distribution of Row Counts per Author/Title/Published Combination", fontsize=14)
plt.grid(axis="y", alpha=0.3)

# Add some statistics to the plot
mean_val = combination_counts.mean()
median_val = combination_counts.median()
plt.axvline(mean_val, color="red", linestyle="--", linewidth=2, label=f"Mean: {mean_val:.1f}")
plt.axvline(
    median_val, color="green", linestyle="--", linewidth=2, label=f"Median: {median_val:.1f}"
)
plt.legend()

plt.tight_layout()
plt.savefig(output_path + "combination_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# Optional: Create a bar chart for the top 20 combinations
plt.figure(figsize=(14, 8))
top_20 = combination_counts.nlargest(20)
# Create labels for the x-axis
labels = [
    f"{auth[:20]}...\n{title[:30]}..." if len(auth) > 20 or len(title) > 30 else f"{auth}\n{title}"
    for auth, title, _ in top_20.index
]

plt.barh(range(len(top_20)), top_20.values, alpha=0.7)
plt.yticks(range(len(top_20)), labels, fontsize=8)
plt.xlabel("Number of Rows", fontsize=12)
plt.ylabel("Author / Title", fontsize=12)
plt.title("Top 20 Combinations by Row Count", fontsize=14)
plt.gca().invert_yaxis()  # Largest at top
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(output_path + "top_20_combinations.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nTop 10 combinations with most rows:")
print(top_20.head(10))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate words per paragraph (assuming 'Text' column contains the text)
# Handle NaN values
deitytext_sorted["words_per_row"] = (
    deitytext_sorted["Text"].fillna("").astype(str).apply(lambda x: len(x.split()))
)

# Remove rows with 0 words (empty text)
words_per_row = deitytext_sorted["words_per_row"][deitytext_sorted["words_per_row"] > 0]

print("Statistics on words per paragraph/row:")
print(words_per_row.describe())

# Create histogram
plt.figure(figsize=(12, 6))
plt.hist(words_per_row, bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("Words per Paragraph/Row", fontsize=12)
plt.ylabel("Frequency", fontsize=12)

plt.title("Distribution of Words per Paragraph (Complete Dataset)", fontsize=14)
plt.grid(axis="y", alpha=0.3)

# Add statistics
mean_val = words_per_row.mean()
median_val = words_per_row.median()
plt.axvline(mean_val, color="red", linestyle="--", linewidth=2, label=f"Mean: {mean_val:.1f}")
plt.axvline(
    median_val, color="green", linestyle="--", linewidth=2, label=f"Median: {median_val:.1f}"
)
plt.legend()

plt.tight_layout()
plt.savefig(output_path + "words_per_paragraph_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# Show top paragraphs with most words
print("\nTop 50 longest paragraphs - word counts:")
top_paragraphs = deitytext_sorted.nlargest(50, "words_per_row")
print(top_paragraphs["words_per_row"])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sort the word counts for CDF
sorted_word_counts = np.sort(words_per_row.values)
# Calculate cumulative probabilities
cumulative_prob = np.arange(1, len(sorted_word_counts) + 1) / len(sorted_word_counts)

# Create CDF plot with log scale
plt.figure(figsize=(12, 6))
plt.plot(sorted_word_counts, cumulative_prob, linewidth=2)
plt.xscale("log")  # Set x-axis to log scale
plt.xlabel("Words per Paragraph (log scale)", fontsize=12)
plt.ylabel("Cumulative Probability", fontsize=12)
plt.title("CDF of Words per Paragraph (Log Scale)", fontsize=14)
plt.grid(True, alpha=0.3, which="both")  # Show grid for both major and minor ticks

# Add percentile lines
median_val = words_per_row.median()
percentile_25 = words_per_row.quantile(0.25)
percentile_75 = words_per_row.quantile(0.75)
percentile_90 = words_per_row.quantile(0.90)
percentile_95 = words_per_row.quantile(0.95)
percentile_99 = words_per_row.quantile(0.99)

plt.axvline(
    median_val,
    color="green",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"Median (50%): {median_val:.1f}",
)
plt.axvline(
    percentile_75,
    color="orange",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"75th percentile: {percentile_75:.1f}",
)
plt.axvline(
    percentile_90,
    color="red",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"90th percentile: {percentile_90:.1f}",
)
plt.axvline(
    percentile_95,
    color="darkred",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"95th percentile: {percentile_95:.1f}",
)
plt.axvline(
    percentile_99,
    color="purple",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"99th percentile: {percentile_99:.1f}",
)

# Add horizontal lines at key probabilities
plt.axhline(0.5, color="green", linestyle=":", linewidth=1, alpha=0.5)
plt.axhline(0.75, color="orange", linestyle=":", linewidth=1, alpha=0.5)
plt.axhline(0.90, color="red", linestyle=":", linewidth=1, alpha=0.5)
plt.axhline(0.95, color="darkred", linestyle=":", linewidth=1, alpha=0.5)
plt.axhline(0.99, color="purple", linestyle=":", linewidth=1, alpha=0.5)

plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(output_path + "words_per_paragraph_cdf_log.png", dpi=300, bbox_inches="tight")
plt.show()

# Print some useful statistics from the CDF
print("\nWord Count CDF Statistics:")
print(f"25th percentile: {percentile_25:.1f} words")
print(f"Median (50th percentile): {median_val:.1f} words")
print(f"75th percentile: {percentile_75:.1f} words")
print(f"90th percentile: {percentile_90:.1f} words")
print(f"95th percentile: {percentile_95:.1f} words")
print(
    f"\n{(cumulative_prob[sorted_word_counts <= 50].max() * 100):.1f}% of paragraphs have ≤50 words"
)
print(
    f"{(cumulative_prob[sorted_word_counts <= 100].max() * 100):.1f}% of paragraphs have ≤100 words"
)
print(
    f"{(cumulative_prob[sorted_word_counts <= 200].max() * 100):.1f}% of paragraphs have ≤200 words"
)
print(
    f"{(cumulative_prob[sorted_word_counts <= 500].max() * 100):.1f}% of paragraphs have ≤500 words"
)

In [ ]:
# Count words per document (row)
words_per_document = deitytext_sorted["Text"].str.split().str.len()

# Add it as a new column to your dataframe
deitytext_sorted["word_count"] = words_per_document

# Group by document and sum word counts
document_summary = (
    deitytext_sorted.groupby(["Primary_Author", "Title", "Published"])
    .agg(
        {
            "word_count": "sum",
            "uuid": "count",  # This gives you number of paragraphs per document
        }
    )
    .rename(columns={"uuid": "paragraph_count", "word_count": "total_words"})
)

print("\nDocument-level word counts:")
print(document_summary.head(10))

# Calculate and display statistics for words per document
print("\nAverage word count per document (statistics):")
print(document_summary["total_words"].describe())

In [ ]:
import pandas as pd

deitytext_sorted["word_count"] = deitytext_sorted["Text"].str.split().str.len()


def split_long_paragraph(text, max_words=1619):
    """Split a paragraph that's longer than max_words into smaller pieces"""
    words = text.split()
    if len(words) <= max_words:
        return [text]

    pieces = []
    current_piece = []

    for word in words:
        current_piece.append(word)
        if len(current_piece) >= max_words:
            pieces.append(" ".join(current_piece))
            current_piece = []

    if current_piece:
        pieces.append(" ".join(current_piece))

    return pieces


def split_text_with_uuids(group, max_words=1619):
    """Split document into chunks while tracking which UUIDs belong to each chunk"""
    chunks = []
    current_chunk_text = []
    current_chunk_uuids = []
    current_chunk_metadata = {
        "pages": [],
        "doctypes": [],
        "cultures": [],
        "regions": [],
        "subregions": [],
        "subsistences": [],
        "ocms": [],
        "ids": [],
        "permalinks": [],
        "image_links": [],
    }
    current_word_count = 0

    for idx, row in group.iterrows():
        paragraph_text = str(row["Text"]) if pd.notna(row["Text"]) else ""
        paragraph_words = paragraph_text.split()
        paragraph_word_count = len(paragraph_words)

        # If this single paragraph is longer than max_words, split it
        if paragraph_word_count > max_words:
            # First, save current chunk if it has content
            if current_chunk_text:
                chunks.append(
                    {
                        "text": " ".join(current_chunk_text),
                        "uuids": current_chunk_uuids.copy(),
                        "metadata": {k: v.copy() for k, v in current_chunk_metadata.items()},
                    }
                )
                current_chunk_text = []
                current_chunk_uuids = []
                current_chunk_metadata = {k: [] for k in current_chunk_metadata}
                current_word_count = 0

            # Split the long paragraph into multiple pieces
            paragraph_pieces = split_long_paragraph(paragraph_text, max_words)

            for piece in paragraph_pieces:
                chunks.append(
                    {
                        "text": piece,
                        "uuids": [row["uuid"]],  # Same UUID for all pieces
                        "metadata": {
                            "pages": [row["Page"]],
                            "doctypes": [row["DocType"]],
                            "cultures": [row["Culture"]],
                            "regions": [row["Region"]],
                            "subregions": [row["Subregion"]],
                            "subsistences": [row["Subsistence"]],
                            "ocms": [row["OCM"]],
                            "ids": [row["IDs"]],
                            "permalinks": [row["Permalink"]],
                            "image_links": [row["Image Link"]],
                        },
                    }
                )

        # If adding this paragraph would exceed max_words, start a new chunk
        elif current_word_count + paragraph_word_count > max_words and current_chunk_text:
            chunks.append(
                {
                    "text": " ".join(current_chunk_text),
                    "uuids": current_chunk_uuids.copy(),
                    "metadata": {k: v.copy() for k, v in current_chunk_metadata.items()},
                }
            )
            # Reset for new chunk
            current_chunk_text = [paragraph_text]
            current_chunk_uuids = [row["uuid"]]
            current_chunk_metadata = {
                "pages": [row["Page"]],
                "doctypes": [row["DocType"]],
                "cultures": [row["Culture"]],
                "regions": [row["Region"]],
                "subregions": [row["Subregion"]],
                "subsistences": [row["Subsistence"]],
                "ocms": [row["OCM"]],
                "ids": [row["IDs"]],
                "permalinks": [row["Permalink"]],
                "image_links": [row["Image Link"]],
            }
            current_word_count = paragraph_word_count

        else:
            # Add paragraph to current chunk
            current_chunk_text.append(paragraph_text)
            current_chunk_uuids.append(row["uuid"])
            current_chunk_metadata["pages"].append(row["Page"])
            current_chunk_metadata["doctypes"].append(row["DocType"])
            current_chunk_metadata["cultures"].append(row["Culture"])
            current_chunk_metadata["regions"].append(row["Region"])
            current_chunk_metadata["subregions"].append(row["Subregion"])
            current_chunk_metadata["subsistences"].append(row["Subsistence"])
            current_chunk_metadata["ocms"].append(row["OCM"])
            current_chunk_metadata["ids"].append(row["IDs"])
            current_chunk_metadata["permalinks"].append(row["Permalink"])
            current_chunk_metadata["image_links"].append(row["Image Link"])
            current_word_count += paragraph_word_count

    # Add final chunk
    if current_chunk_text:
        chunks.append(
            {
                "text": " ".join(current_chunk_text),
                "uuids": current_chunk_uuids,
                "metadata": {k: v for k, v in current_chunk_metadata.items()},
            }
        )

    return chunks


def join_unique(lst):
    unique_vals = []
    seen = set()
    for item in lst:
        if pd.notna(item) and item not in seen:
            unique_vals.append(str(item))
            seen.add(item)
    return "; ".join(unique_vals) if unique_vals else ""


def join_list(lst):
    return "; ".join([str(item) for item in lst if pd.notna(item)])


condensed_rows = []
def_id_counter = 0

for (author, title, published), group in deitytext_sorted.groupby(
    ["Primary_Author", "Title", "Published"]
):
    chunks = split_text_with_uuids(group, max_words=1619)

    for chunk_data in chunks:
        condensed_rows.append(
            {
                "def_id": def_id_counter,
                "old_uuids": join_list(chunk_data["uuids"]),
                "Primary_Author": author,
                "Title": title,
                "Published": published,
                "Page": join_list(chunk_data["metadata"]["pages"]),
                "DocType": join_unique(chunk_data["metadata"]["doctypes"]),
                "Culture": join_unique(chunk_data["metadata"]["cultures"]),
                "Region": join_unique(chunk_data["metadata"]["regions"]),
                "Subregion": join_unique(chunk_data["metadata"]["subregions"]),
                "Subsistence": join_unique(chunk_data["metadata"]["subsistences"]),
                "OCM": join_unique(chunk_data["metadata"]["ocms"]),
                "IDs": join_unique(chunk_data["metadata"]["ids"]),
                "Permalink": chunk_data["metadata"]["permalinks"][0]
                if chunk_data["metadata"]["permalinks"]
                else "",
                "Text": chunk_data["text"],
                "Image Link": chunk_data["metadata"]["image_links"][0]
                if chunk_data["metadata"]["image_links"]
                else "",
            }
        )
        def_id_counter += 1

deities_information_condensed = pd.DataFrame(condensed_rows)
deities_information_condensed.to_csv(
    output_path + "deities_information_condensed.csv", index=False, encoding="utf-8-sig"
)

print(f"Original file rows: {len(deitytext_sorted)}")
print(f"Condensed file rows: {len(deities_information_condensed)}")
print(f"def_id range: 0 to {def_id_counter - 1}")

# Check for any chunks that exceed the limit
deities_information_condensed["word_count"] = (
    deities_information_condensed["Text"].str.split().str.len()
)
over_limit = deities_information_condensed[deities_information_condensed["word_count"] > 1619]
print(f"Chunks exceeding 1619 words: {len(over_limit)}")
print(f"Max word count: {deities_information_condensed['word_count'].max()}")

In [ ]:
# First, add word count to the condensed dataframe
deities_information_condensed["word_count"] = (
    deities_information_condensed["Text"].str.split().str.len()
)

# Find documents with word count between 1500 and 1619
filtered_docs = deities_information_condensed[
    (deities_information_condensed["word_count"] >= 1000)
    & (deities_information_condensed["word_count"] <= 1619)
]

# Take the first one as a sample
sample_doc = filtered_docs.sample(9)

# Save to CSV
sample_doc.to_csv(
    output_path + "sample_condensed_document_2.csv", index=False, encoding="utf-8-sig"
)

print("Sample document saved!")
print(f"Word count: {sample_doc['word_count'].values[0]}")
print("\nSample document details:")
print(f"Author: {sample_doc['Primary_Author'].values[0]}")
print(f"Title: {sample_doc['Title'].values[0]}")
print(f"Number of original paragraphs: {len(sample_doc['old_uuids'].values[0])}")
print("\nFirst 200 characters of text:")
print(sample_doc["Text"].values[0][:200])

In [ ]:
import re


# Clean the Text column
def clean_text(text):
    if pd.isna(text):
        return text

    # Remove "Load in context"
    text = text.replace("Load in context", "")

    # Remove insert_drive_file followed by any numbers
    text = re.sub(r"insert_drive_file\d+", "", text)

    # Remove extra whitespace that might be left over
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply cleaning to the Text column
deitytext_sorted["Text"] = deitytext_sorted["Text"].apply(clean_text)

print("Text column cleaned!")
print("\nSample of cleaned text:")
print(deitytext_sorted["Text"].head(3))

In [ ]:
# Clean the Text column in sample_doc
def clean_text(text):
    if pd.isna(text):
        return text

    # Remove "Load in context"
    text = re.sub(r"\S*Load in context\S*", "", text, flags=re.IGNORECASE)

    # Remove insert_drive_file followed by any numbers
    text = re.sub(r"insert_drive_file\d+", "", text)

    # Remove extra whitespace that might be left over
    text = re.sub(r"\s+", " ", text).strip()

    return text


sample_doc = pd.read_csv(output_path + "sample_condensed_document_2.csv", encoding="utf-8-sig")
# Apply cleaning to sample_doc
sample_doc["Text"] = sample_doc["Text"].apply(clean_text)

# Save the cleaned sample
sample_doc.to_csv(
    output_path + "sample_condensed_document_cleaned_2.csv", index=False, encoding="utf-8-sig"
)

print("Sample document cleaned and saved!")
print("\nCleaned text preview (first 300 chars):")
print(sample_doc["Text"].values[0][:300])